**IMPORTS NECESARIOS Y CONFIGURACIÓN GLOBAL**

Para llevar a cabo la detección y lectura de matrículas, inicialmente se importan las librerías necesarias. En esta versión el OCR clásico (PaddleOCR) se sustituye por un **modelo de visión-lenguaje (VLM)**, `PaddleOCR-VL`.

In [1]:
import os
import csv
from functools import partial
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics import YOLO

from utils_matriculas import (
    GT_KEYS,
    cargar_gt,
    evaluar_dataset,
    get_ocr_vlm,
    iou,
    leer_cronometrado,
    leer_placa_vlm,
    lev,
    procesar_video_yolo,
    recortar,
)

**INICIALIZACIÓN GLOBAL DE MODELOS**

Seleccionaremos el detector de matrículas entrenado `best-yolo.pt`.

In [2]:
# Ruta al detector entrenado de placas
WEIGHTS_PLACAS = "weights/best-yolo.pt"

# Inicializacion del detector de matriculas compartido
detector_placas = YOLO(WEIGHTS_PLACAS)


**CARGA DEL MODELO VLM**

La carga se delega en `get_ocr_vlm` de `utils_matriculas.py`, que devuelve el par `(modelo, processor)` del VLM y lo guarda en caché para cargarlo una sola vez. Aquí solo se fija el modelo `VLM_ID` y el dispositivo `DEVICE`.

In [3]:
# Identificador del modelo VLM y dispositivo de ejecucion
VLM_ID = "PaddlePaddle/PaddleOCR-VL"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Inicializamos el VLM globalmente: devuelve (modelo, processor), o None si falla la carga
ocr_global = get_ocr_vlm(DEVICE, VLM_ID)

c:\Users\vidal\miniconda3\envs\VC_P4\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\vidal\miniconda3\envs\VC_P4\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vidal\.cache\huggingface\hub\models--PaddlePaddle--PaddleOCR-VL. Caching files will still work but in a degraded ve

**UTILIDADES COMUNES: GEOMETRÍA, CADENAS Y MANIPULACIÓN DE CAJAS**

Las funciones auxiliares compartidas (`norm`, `lev`, `iou`, `centro`, `diag`, `recortar`) viven ahora en `utils_matriculas.py`. En este cuaderno se importan las que se usan directamente (`lev`, `iou`, `recortar`); el resto las utilizan internamente las demás funciones de utils.

**LECTURA DE MATRÍCULAS CON VLM (`leer_placa`)**

La lectura está implementada en `leer_placa_vlm` (`utils_matriculas.py`): mismo preprocesamiento que en la versión con PaddleOCR y lectura delegada en el VLM mediante una función fija que pide solo los caracteres de la placa.

Como un VLM no devuelve una confianza por lectura, se estima como la **media de la probabilidad de los tokens generados**, de modo que conserva la firma `(texto, confianza)`. Aquí se fijan con `partial` el dispositivo y el modelo ya cargado, para que `leer_placa(crop)` siga recibiendo solo el recorte.

In [4]:
# Lee la placa de una imagen
leer_placa = partial(leer_placa_vlm, device=DEVICE, ocr_inst=ocr_global)

**TAREA 1: ADQUISICIÓN Y EVALUACIÓN DE MÉTRICAS DE LECTURA CON VLM**

Evaluamos el rendimiento de la detección de matrículas:
- **Exact Match**: Porcentaje de matrículas donde la cadena leída coincide al 100% con el texto real.
- **Acierto por carácter**: Precisión media basada en la distancia normalizada: $1 - \min(1, \text{lev} / \text{longitud})$.
- **Tiempos de ejecución (`perf_counter`)**: Se mide el tiempo medio empleado por detección tanto en localización como en reconocimiento, así como el coste temporal total.
- **Análisis de falsos positivos**: Se diferencian las detecciones sin coincidencia con ninguna caja real de aquellas que coinciden con una placa anotada pero sin texto legible (`None`/vacía).

**CONFIGURACIÓN DEL DATASET DE EVALUACIÓN**

Rutas, umbrales y carpeta de salida donde se guardan el CSV de resultados y los recortes con error.

In [5]:
# Configuracion de rutas y parametros del dataset
BASE = Path("plates/raw-dataset")
IMAGENES = BASE / "images"
ANOTACIONES = BASE / "annotations"
SPLIT = "train"
MAX_IMAGENES = 200     # None para evaluar el dataset completo
CONF_METRICAS = 0.25
IOU_MATCH = 0.5
PAD = 0.05
IMGSZ_METRICAS = 960
EXTS = {".jpg", ".jpeg", ".png"}

SALIDA = Path("vlm_eval")
SALIDA.mkdir(exist_ok=True)
(SALIDA / "errores").mkdir(exist_ok=True)


**FUNCIONES AUXILIARES DE EVALUACIÓN**

La carga del Ground Truth desde los JSON (`cargar_gt`) y el resumen de exact match y acierto por carácter (`resumen_metricas`) están en `utils_matriculas.py`. `cargar_gt` recibe la carpeta de anotaciones como argumento, así que aquí se fija con `partial` (`gt_dataset`) para usarla con `ANOTACIONES`.

In [6]:
# cargar_gt (utils) recibe la carpeta de anotaciones y las claves como argumentos: las fijamos aqui
gt_dataset = partial(cargar_gt, anotaciones_dir=ANOTACIONES, gt_keys=GT_KEYS)

La función `evaluar_dataset` está en `utils_matriculas.py` y es independiente del detector. Emplea `perf_counter` para registrar con alta precisión los tiempos medios de lectura por detección, clasifica los posibles falsos positivos geométricos frente a placas anotadas con texto `None`, y genera tanto el archivo `ocr_resultados.csv` como el guardado de recortes erróneos en disco.

Aquí solo se define `detectar_yolo`, que adapta el detector YOLO a lo que espera `evaluar_dataset`: recibe una imagen y devuelve una lista de `(caja_xyxy, confianza)`.

In [7]:
# Adaptador del detector YOLO para evaluar_dataset
def detectar_yolo(img, detector=detector_placas):
    r = detector.predict(img, imgsz=IMGSZ_METRICAS, conf=CONF_METRICAS, verbose=False)[0]
    return [(b.xyxy[0].tolist(), float(b.conf[0])) for b in r.boxes]

**EJECUCIÓN DE LA EVALUACIÓN**

Se evalúa un subconjunto de imágenes del dataset (`MAX_IMAGENES`). Con el VLM esta celda es notablemente más lenta que con PaddleOCR.

In [8]:
# Ejecucion de la evaluacion de metricas
if IMAGENES.exists():
    lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
    if MAX_IMAGENES:
        lista = lista[:MAX_IMAGENES]
    evaluar_dataset(lista, detectar_yolo, leer_placa, gt_dataset, SALIDA, "errores",
                    "ocr_resultados.csv", "YOLO", iou_match=IOU_MATCH, pad=PAD,
                    gt_keys=GT_KEYS, delimiter=",")
else:
    print(f"Directorio {IMAGENES} no encontrado. Comprueba la ruta del dataset.")



Imágenes evaluadas: 200 | placas anotadas: 407 (con texto: 407) | detecciones YOLO: 367
--- TIEMPOS MEDIOS DE LECTURA POR DETECCIÓN (perf_counter) ---
OCR a partir de detección YOLO:     650.17 ms / detección
OCR a partir de recorte GT (techo): 764.55 ms / recorte anotado

--- RESUMEN DE EXACTITUD OCR ---
OCR sobre detecciones de YOLO      n=188   exact match=0.798 | acierto por carácter=0.922
OCR sobre recorte anotado (techo)  n=210   exact match=0.729 | acierto por carácter=0.835
Placas evaluadas encontradas por YOLO: 188/407
Falsos positivos geométricos (IoU < 0.5): 108
Detecciones en placas anotadas sin texto (None): 71

CSV: vlm_eval\ocr_resultados.csv
Recortes con error: vlm_eval\errores


**TAREA 1: EVALUACIÓN DEL VLM CON CORRECCIÓN POR POSICIÓN**

Se añade una segunda evaluación que reutiliza el detector, el VLM y las utilidades ya definidas (`norm`, `lev`, `iou`, `recortar`, `cargar_gt`, `leer_placa`). Respecto a `evaluar_dataset` incorpora:
- **Lectura cruda vs. lectura corregida**: se mide el VLM tal cual y tras aplicar una corrección por posición (`corregir`).
- **Tiempo por lectura en milisegundos**: media y mediana, con una lectura de calentamiento previa, ya que la primera inferencia del VLM es notablemente más lenta.

**CORRECCIÓN POR POSICIÓN Y LECTURA CRONOMETRADA**

`corregir` y `leer_cronometrado` están en `utils_matriculas.py`. `corregir` aprovecha que en las matrículas españolas actuales cada posición admite un solo tipo de carácter: en las posiciones numéricas se sustituyen letras confundibles y en las posiciones de letras se sustituyen dígitos confundibles. Solo se aplica a lecturas de 7 caracteres y únicamente se acepta el resultado si cumple el formato; en caso contrario se devuelve la lectura sin tocar. `leer_cronometrado(crop, leer_fn)` envuelve a la función de lectura que se le pase (aquí `leer_placa`) midiendo su tiempo con `perf_counter`.

**FUNCIONES DE EVALUACIÓN CON LECTURA CRUDA Y CORREGIDA**

`resumen_metricas_vlm` muestra exact match, acierto por carácter ($1 - \min(1, \text{lev} / \text{longitud real})$) y tiempo medio y mediano para la columna indicada (`ocr_crudo` u `ocr_corregido`). `evaluar_dataset_vlm` evalúa tanto sobre las detecciones de YOLO (resultado de extremo a extremo) como sobre el recorte exacto de la anotación (techo del OCR, que aísla el error del detector) y guarda el CSV con separador `;`.

In [9]:
# Muestra exact match, acierto por caracter y tiempos para la columna indicada
def resumen_metricas_vlm(titulo, rows, col):
    if not rows:
        return
    exacto = np.mean([r["gt"] == r[col] for r in rows])
    por_car = np.mean([1 - min(1, lev(r["gt"], r[col]) / max(len(r["gt"]), 1)) for r in rows])
    tiempos = [r["ms"] for r in rows]
    print(f"{titulo:<40} n={len(rows):<5} exacto={exacto:.3f} | por carácter={por_car:.3f} "
          f"| {np.mean(tiempos):.0f} ms de media ({np.median(tiempos):.0f} de mediana)")


def evaluar_dataset_vlm(lista_imagenes, detector=detector_placas):
    """Evalua el VLM (crudo y corregido) sobre detecciones YOLO y sobre recortes anotados (techo)."""
    leer_cronometrado(np.zeros((30, 100, 3), np.uint8), leer_placa)   # calentamiento: la 1a lectura es mas lenta

    filas = []
    n_gt = n_gt_texto = n_detectadas = 0

    for img_path in lista_imagenes:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        gts = gt_dataset(img_path, w, h)
        n_gt += len(gts)
        n_gt_texto += sum(t is not None for _, t in gts)

        # Inferencia de deteccion de placas
        r = detector.predict(img, imgsz=IMGSZ_METRICAS, conf=CONF_METRICAS, verbose=False)[0]
        preds = [(b.xyxy[0].tolist(), float(b.conf[0])) for b in r.boxes]
        n_detectadas += len(preds)

        # 1) OCR sobre lo que detecta YOLO (resultado de extremo a extremo)
        for box, det_conf in preds:
            rec = recortar(img, box, pad=PAD)
            cruda, corregida, ms = leer_cronometrado(rec, leer_placa)

            mejor_iou, gt_txt = 0.0, None
            for gbox, gtxt in gts:
                v = iou(box, gbox)
                if v > mejor_iou:
                    mejor_iou, gt_txt = v, gtxt
            if mejor_iou < IOU_MATCH or gt_txt is None or gt_txt == "NONE":
                continue

            filas.append(dict(imagen=img_path.name, origen="deteccion", gt=gt_txt, ocr_crudo=cruda,
                              ocr_corregido=corregida, ms=round(ms, 1), det_conf=round(det_conf, 3),
                              iou=round(mejor_iou, 3)))
            if gt_txt != cruda:
                cv2.imwrite(str(SALIDA / "errores-yolo" / f"{len(filas):05d}_gt-{gt_txt}_ocr-{cruda}.jpg"), rec)

        # 2) OCR sobre el recorte exacto de la anotacion (techo del OCR, aisla el error de YOLO)
        for gbox, gtxt in gts:
            if gtxt is None or gtxt == "NONE":
                continue
            cruda, corregida, ms = leer_cronometrado(recortar(img, gbox, pad=PAD), leer_placa)
            filas.append(dict(imagen=img_path.name, origen="gt_recorte", gt=gtxt, ocr_crudo=cruda,
                              ocr_corregido=corregida, ms=round(ms, 1), det_conf=None, iou=1.0))

    # Guardar resultados en CSV (separador ;)
    csv_vlm = SALIDA / "yolo_resultados_vlm.csv"
    with open(csv_vlm, "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=list(filas[0].keys()) if filas else ["vacio"], delimiter=";")
        wr.writeheader()
        wr.writerows(filas)

    print(f"\nImágenes: {len(lista_imagenes)} | placas anotadas: {n_gt} | detecciones YOLO: {n_detectadas}")
    if n_gt_texto == 0:
        print(f"Las anotaciones no traen texto (claves probadas: {GT_KEYS}).")
    else:
        det = [r for r in filas if r["origen"] == "deteccion"]
        techo = [r for r in filas if r["origen"] == "gt_recorte"]
        resumen_metricas_vlm("VLM sobre detecciones de YOLO", det, "ocr_crudo")
        resumen_metricas_vlm("VLM sobre recorte anotado (techo)", techo, "ocr_crudo")
        resumen_metricas_vlm("VLM + corrección, detecciones", det, "ocr_corregido")
        resumen_metricas_vlm("VLM + corrección, recorte anotado", techo, "ocr_corregido")
        print(f"Placas anotadas encontradas por YOLO: {len(det)}/{n_gt_texto}")
    print(f"\nCSV: {csv_vlm}\nRecortes con error: {SALIDA / 'errores'}")
    return filas

**EJECUCIÓN DE LA EVALUACIÓN CON CORRECCIÓN**

Usa el mismo subconjunto de imágenes (`MAX_IMAGENES`) y la misma configuración de rutas que la evaluación anterior. Si el directorio del dataset no se encuentra, se muestra un aviso: hay que revisar `BASE` en la celda de configuración del dataset.

In [10]:
# Ejecucion de la evaluacion con lectura cruda y corregida
if IMAGENES.exists():
    lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
    if MAX_IMAGENES:
        lista = lista[:MAX_IMAGENES]
    filas_vlm = evaluar_dataset_vlm(lista)
else:
    print(f"Directorio {IMAGENES} no encontrado. Comprueba la ruta del dataset (variable BASE).")


Imágenes: 200 | placas anotadas: 407 | detecciones YOLO: 367
VLM sobre detecciones de YOLO            n=188   exacto=0.798 | por carácter=0.922 | 749 ms de media (727 de mediana)
VLM sobre recorte anotado (techo)        n=210   exacto=0.729 | por carácter=0.835 | 767 ms de media (719 de mediana)
VLM + corrección, detecciones            n=188   exacto=0.835 | por carácter=0.931 | 749 ms de media (727 de mediana)
VLM + corrección, recorte anotado        n=210   exacto=0.757 | por carácter=0.841 | 767 ms de media (719 de mediana)
Placas anotadas encontradas por YOLO: 188/407

CSV: vlm_eval\yolo_resultados_vlm.csv
Recortes con error: vlm_eval\errores


**TAREA 2: DETECCIÓN, TRACKING Y LECTURA DE MATRÍCULAS EN VÍDEO (VLM)**

En este módulo se procesa una secuencia de vídeo realizando:
- **Seguimiento con ByteTrack**: Detección y seguimiento continuo de personas y vehículos (`persona`, `coche`, `moto`, `guagua`, `camion`).
- **Filtrado y validación sintáctica**: Verificación de las matrículas leídas mediante expresiones regulares de formatos de placas europeos.
- **Fusión de tracklets**: Recuperación de identificadores ante oclusiones temporales, tanto para vehículos en movimiento como para vehículos estacionados.
- **Renderizado y exportación**: Generación del vídeo anotado con los identificadores y recuadros de matrícula, junto con el archivo `resultado.csv` detallando las detecciones por fotograma.

**CONFIGURACIÓN DE RUTAS, CLASES Y PALETA**

Rutas del vídeo de entrada y de salida (`resultado.mp4` y `resultado.csv` se guardan en la carpeta `YOLO_VLM_result`), mapeo de clases COCO y colores de dibujo.

In [11]:
# Configuracion de rutas de video
filename = "C0142.MP4"
# Carpeta de resultados: se crea en el directorio de trabajo del cuaderno
carpeta = os.path.abspath("YOLO_VLM_result")
os.makedirs(carpeta, exist_ok=True)
output_path = os.path.join(carpeta, "resultado.mp4")
csv_path = os.path.join(carpeta, "resultado.csv")

# Mapeo de clases y paleta visual
nombre = {0: "persona", 2: "coche", 3: "moto", 5: "guagua", 7: "camion"}
colores = {0: (0, 255, 0), 2: (0, 0, 255), 3: (255, 0, 255),
           5: (0, 255, 255), 7: (255, 255, 0)}

COLOR_PLACA = (0, 165, 255)
ESCALA_TEXTO_PLACA = 0.35
CONF_PLACA = 0.4
MIN_FRAMES_PLACA = 5


**PARÁMETROS DE LECTURA**

Control de cuándo se lanza la lectura sobre cada vehículo. Los patrones sintácticos de matrículas europeas (`PATRONES_MATRICULA`) y la función `matricula_valida` están en `utils_matriculas.py`.

In [12]:
# Parametros del OCR sobre video
OCR_CADA = 10
MIN_ANCHO_OCR = 40
MIN_CHARS_PLACA = 4
OCR_SCORE_OK = 0.5
OCR_MAX_INTENTOS = 3

**PARÁMETROS DE TRACKING**

Parámetros del voto de clase y configuración del archivo de ByteTrack. Los parámetros de fusión de tracklets (`FAMILIA`, `PARAM_FUSION`, `GAP_ESTATICO_SEG`, `MIN_FRAMES_ESTATICO`, `IOU_ESTATICO`, `MAYORIA_PESADOS`) se definen en `utils_matriculas.py`.

In [13]:
# Parametros de tracking
MIN_FRAMES = 10
MAYORIA_PESADOS = 0.65
PESO_BORDE = 0.3
MARGEN_BORDE = 5

# Configuracion del archivo de ByteTrack
os.makedirs("./YOLO", exist_ok=True)
with open("./YOLO/mi_bytetrack.yaml", "w") as f:
    f.write("tracker_type: bytetrack\ntrack_high_thresh: 0.25\ntrack_low_thresh: 0.1\n"
            "new_track_thresh: 0.25\ntrack_buffer: 140\nmatch_thresh: 0.8\nfuse_score: True\n")


Las funciones de validación, tracking y fusión (`matricula_valida`, `prediccion_posicion`, `es_estatico`, `fusionar_tracklets`, `decidir_clase`) están en `utils_matriculas.py` y las utiliza `procesar_video_yolo`.

Finalmente, `procesar_video_yolo` (en `utils_matriculas.py`) integra las dos pasadas:
1. **Pasada 1**: Ejecución de inferencia sobre el vídeo, seguimiento con ByteTrack, detección de matrículas mediante `detector_placas`, lectura con el VLM mediante `leer_placa` y fusión temporal de identidades.
2. **Pasada 2**: Renderizado del vídeo final `resultado.mp4` superponiendo las cajas y matrículas leídas, y volcado a `resultado.csv`.

Los parámetros definidos en este cuaderno se le pasan como argumentos.

**EJECUCIÓN DEL DE VÍDEO**

In [14]:
if os.path.exists(filename):
    procesar_video_yolo(
        filename, output_path, csv_path, detector_placas, leer_placa, nombre, colores,
        conf_placa=CONF_PLACA, min_frames_placa=MIN_FRAMES_PLACA, color_placa=COLOR_PLACA,
        escala_texto_placa=ESCALA_TEXTO_PLACA, ocr_cada=OCR_CADA, min_ancho_ocr=MIN_ANCHO_OCR,
        min_chars_placa=MIN_CHARS_PLACA, ocr_score_ok=OCR_SCORE_OK,
        ocr_max_intentos=OCR_MAX_INTENTOS, min_frames=MIN_FRAMES,
        peso_borde=PESO_BORDE, margen_borde=MARGEN_BORDE,
        track_kwargs=dict(quantize=16, verbose=False),
        predict_kwargs=dict(quantize=16, verbose=False),
        etiqueta_ocr="VLM", ocr_solo_en_placas=True,
        max_ancho_salida=1920, mostrar_progreso=True,
    )
else:
    print(f"Vídeo {filename} no encontrado. Comprueba la ruta.")

Iniciando Pasada 1: tracking, detección y lectura OCR en vídeo...


KeyboardInterrupt: 